In [1]:
%load_ext autoreload
%autoreload 2

# IPL Data-Backed Analysis: Validating the Blog Post Claims

This notebook provides a comprehensive data analysis to validate all statistical claims made in the IPL blog post "The Death of Caution: How the IPL Transformed Cricket's Most Conservative League into a Slogfest".

## Analysis Goals

1. **Batting Strike Rate Trends**: Validate 7.24% increase from 2008-2015 to 2016-2024
2. **Six-Hitting Growth**: Validate 361% increase from 3.1 to 14.3 sixes/match
3. **Phase-Wise Scoring**: Validate powerplay (+28%), middle, and death over trends
4. **Bowling Metrics**: Validate economy rate (+7.37%) and dot-ball suppression changes
5. **Impact Player Rule**: Validate 1.4 sixes/match increase post-2023
6. **Venue Impact**: Analyze boundary size correlation with six frequency
7. **Statistical Testing**: Run hypothesis tests with FDR correction
8. **Projections**: Validate six-hitting projections to 2028

## Data Source

Ball-by-ball IPL data from 2008-2025, containing:
- 1,169 matches
- 278,205 ball records
- Complete batting and bowling statistics


In [2]:
# Import dependencies
import pandas as pd
import numpy as np
import sys
import warnings
from pathlib import Path

# Set display options
pd.set_option('display.max_columns', 10)
pd.set_option('display.max_rows', 50)
np.set_printoptions(precision=4, suppress=True)

warnings.filterwarnings('ignore')

# Import local package
sys.path.insert(0,  str(Path(".").resolve().parent / 'src'))

from data_loader import IPLDataLoader, load_ipl_data
from metrics import (
    calculate_season_strike_rates,
    calculate_phase_scoring,
    calculate_six_rates,
    calculate_bowling_metrics,
    calculate_wicket_metrics,
    compare_eras,
    perform_statistical_tests,
    calculate_projected_sixes,
    analyze_venue_impact
)
from visualization import (
    create_strike_rate_trend,
    create_sixes_growth_chart,
    create_phase_scoring_chart,
    create_bowling_metrics_chart,
    create_venue_impact_chart,
    create_statistical_tests_chart,
    create_q_value_chart,
    create_projection_chart
)

print("Dependencies loaded successfully!")

Dependencies loaded successfully!


---
## Step 1: Load and Prepare Data

In [3]:
# Load IPL data
print("Loading IPL data...")
data_loader = IPLDataLoader(data_path=None)  # Will use default path
data = data_loader.load(verbose=True)
clean_data = data_loader.clean()

print(f"\nData shape: {clean_data.shape}")
print(f"\nFirst 5 rows:")
clean_data.head()

Loading IPL data...
Loaded 289914 records from C:\Users\abhij\Code\quortol\analysis\ipl\data\IPL.csv
Columns: ['ID', 'Innings', 'Overs', 'BallNumber', 'Batter', 'Bowler', 'NonStriker', 'ExtraType', 'BatsmanRun', 'ExtrasRun', 'TotalRun', 'IsWicketDelivery', 'PlayerOut', 'Kind', 'FieldersInvolved', 'BattingTeam']

Cleaned data shape: (289914, 25)

Seasons available: [np.int32(2008), np.int32(2009), np.int32(2010), np.int32(2011), np.int32(2012), np.int32(2013), np.int32(2014), np.int32(2015), np.int32(2016), np.int32(2017), np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025), np.int32(2026)]
Total matches: 1219
Total sixes: 15293

Data shape: (289914, 25)

First 5 rows:


,match_id,inning,over,ball_number,batsman,...,bowling_team,is_boundary,is_six,is_dot,is_legal
0,981001,1,0,1,RV Uthappa,...,Kolkata Knight Riders,False,False,True,1
1,981001,1,0,2,RV Uthappa,...,Kolkata Knight Riders,False,False,False,1
2,981001,1,0,3,G Gambhir,...,Kolkata Knight Riders,False,False,True,1
3,981001,1,0,4,G Gambhir,...,Kolkata Knight Riders,False,False,False,1
4,981001,1,0,5,RV Uthappa,...,Kolkata Knight Riders,False,False,False,1


In [4]:
# Data summary
print("Data Summary:")
print(f"Total records: {len(clean_data)}")
print(f"Unique seasons: {clean_data['season'].nunique()}")
print(f"Seasons covered: {sorted(clean_data['season'].unique())}")
print(f"\nTeams:")
print(f"Batting teams: {clean_data['batting_team'].nunique()}")
print(f"Bowling teams: {clean_data['bowling_team'].nunique()}")
print(f"\nTotal sixes: {clean_data['is_six'].sum()}")
print(f"Total matches: {clean_data['match_id'].nunique()}")

Data Summary:
Total records: 289914
Unique seasons: 19
Seasons covered: [np.int32(2008), np.int32(2009), np.int32(2010), np.int32(2011), np.int32(2012), np.int32(2013), np.int32(2014), np.int32(2015), np.int32(2016), np.int32(2017), np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025), np.int32(2026)]

Teams:
Batting teams: 17
Bowling teams: 17

Total sixes: 15293
Total matches: 1219


---
## Step 2: Validate Batting Strike Rate Trends (Claim: +7.24% increase)

**Expected**: Strike rates increased from 102.15 (early era) to 109.55 (late era)


In [5]:
# Calculate season strike rates
print("Calculating season strike rates...")
season_sr = clean_data.groupby('season').agg({
    'batsman_runs': 'sum',
    'is_legal': 'sum'
}).reset_index()

season_sr['strike_rate'] = (season_sr['batsman_runs'] * 100) / (season_sr['is_legal'] + 0.001)

print("\nStrike Rates by Season:")
print(season_sr[['season', 'batsman_runs', 'is_legal', 'strike_rate']].to_string(index=False))

# Era comparison
early_sr = season_sr[(season_sr['season'] >= 2008) & (season_sr['season'] <= 2015)]['strike_rate'].mean()
late_sr = season_sr[(season_sr['season'] >= 2016) & (season_sr['season'] <= 2024)]['strike_rate'].mean()
change = ((late_sr - early_sr) / early_sr) * 100

print(f"\n*** Early Era (2008-2015) Avg SR: {early_sr:.2f}")
print(f"*** Late Era (2016-2024) Avg SR: {late_sr:.2f}")
print(f"*** Change: {change:.2f}% (Claim: +7.24%)")

Calculating season strike rates...

Strike Rates by Season:
 season  batsman_runs  is_legal  strike_rate
   2008         16809     13489   124.612638
   2009         15376     13606   113.008958
   2010         17754     14498   122.458262
   2011         19928     17013   117.133950
   2012         21323     17767   120.014627
   2013         21487     18177   118.209819
   2014         17943     14300   125.475516
   2015         17427     13652   127.651617
   2016         17962     14096   127.426211
   2017         17920     13862   129.274266
   2018         19098     14286   133.683317
   2019         18607     14312   130.009773
   2020         18566     14559   127.522486
   2021         17727     14425   122.890806
   2022         23052     17912   128.695839
   2023         24428     17863   136.751938
   2024         24657     17103   144.167681
   2025         25309     17285   146.421744
   2026         17385     11709   148.475519

*** Early Era (2008-2015) Avg SR: 121.0

In [6]:
# Visualize strike rate trend
fig = create_strike_rate_trend(season_sr, output_path='figures/strike_rate_trend.png')
print("Created: figures/strike_rate_trend.png")

Created: figures/strike_rate_trend.png


In [7]:
fig

---
## Step 3: Validate Six-Hitting Growth (Claim: 361% increase)

**Expected**: Sixes increased from 3.1 to 14.3 per match (361% increase)


In [8]:
# Calculate sixes by season
print("Calculating six-hitting trends...")
season_sixes = clean_data.groupby('season').agg({
    'is_six': 'sum',
    'match_id': 'first'
}).reset_index()

# Count unique matches per season
season_matches = clean_data.groupby('season').nunique()['match_id'].reset_index()
season_matches.rename(columns={'match_id': 'matches'}, inplace=True)

season_sixes = season_sixes.merge(season_matches, on='season')
season_sixes['sixes_per_match'] = season_sixes['is_six'] / (season_sixes['matches'] + 0.001)

print("\nSix-Hitting by Season:")
print(season_sixes[['season', 'is_six', 'matches', 'sixes_per_match']].to_string(index=False))

# Era comparison
early_sixes = season_sixes[(season_sixes['season'] >= 2008) & (season_sixes['season'] <= 2015)]['sixes_per_match'].mean()
late_sixes = season_sixes[(season_sixes['season'] >= 2016) & (season_sixes['season'] <= 2024)]['sixes_per_match'].mean()
change = ((late_sixes - early_sixes) / early_sixes) * 100

print(f"\n*** Early Era Avg Sixes/M: {early_sixes:.2f}")
print(f"*** Late Era Avg Sixes/M: {late_sixes:.2f}")
print(f"*** Change: {change:.1f}% (Claim: +361%)")

Calculating six-hitting trends...

Six-Hitting by Season:
 season  is_six  matches  sixes_per_match
   2008     623       58        10.741194
   2009     508       57         8.912124
   2010     587       60         9.783170
   2011     639       73         8.753305
   2012     733       74         9.905272
   2013     681       76         8.960408
   2014     715       60        11.916468
   2015     692       59        11.728615
   2016     639       60        10.649823
   2017     706       59        11.965899
   2018     872       60        14.533091
   2019     786       60        13.099782
   2020     736       60        12.266462
   2021     687       60        11.449809
   2022    1062       74        14.351157
   2023    1124       74        15.188984
   2024    1261       71        17.760313
   2025    1302       74        17.594357
   2026     940       50        18.799624

*** Early Era Avg Sixes/M: 10.09
*** Late Era Avg Sixes/M: 13.47
*** Change: 33.6% (Claim: +361%)


In [9]:
# Visualize sixes growth
fig = create_sixes_growth_chart(season_sixes, output_path='figures/sixes_growth.png')
print("Created: figures/sixes_growth.png")

Created: figures/sixes_growth.png


In [10]:
fig

---
## Step 4: Validate Phase-Wise Scoring (Claim: Powerplay +28% increase)

**Expected**: Powerplay 7.12→9.12 runs/over, Middle 7.52→8.60, Death 9.42→10.36


In [11]:
# Calculate phase-wise runs per over
print("Calculating phase-wise scoring...")

# Add phase column
clean_data_with_phase = clean_data.copy()
clean_data_with_phase['phase'] = clean_data_with_phase['over'].apply(
    lambda x: 'powerplay' if x <= 5 else ('middle' if x <= 13 else 'death')
)

# Aggregate by phase and season
phase_data = clean_data_with_phase.groupby(['season', 'phase']).agg({
    'batsman_runs': 'sum',
    'is_legal': 'sum'
}).reset_index()

phase_data['runs_per_over'] = phase_data['batsman_runs'] / (phase_data['is_legal'] / 6 + 0.001)

print("\nPhase-wise Runs Per Over by Season:")
phase_display = phase_data[['season', 'phase', 'runs_per_over']].pivot(index='season', columns='phase', values='runs_per_over')
print(phase_display.to_string())

# Era summary by phase
print("\n*** Phase Summary by Era:")
for phase in ['powerplay', 'middle', 'death']:
    phase_early = phase_data[(phase_data['phase'] == phase) & (phase_data['season'] <= 2015)]['runs_per_over'].mean()
    phase_late = phase_data[(phase_data['phase'] == phase) & (phase_data['season'] >= 2016)]['runs_per_over'].mean()
    change = ((phase_late - phase_early) / phase_early) * 100
    print(f"  {phase}: {phase_early:.2f} → {phase_late:.2f} (+{change:.1f}%)")

Calculating phase-wise scoring...

Phase-wise Runs Per Over by Season:
phase      death    middle  powerplay
season                               
2008    8.942408  7.112568   6.775994
2009    8.091671  6.182536   6.407193
2010    8.431896  6.914491   6.952142
2011    8.226712  6.780302   6.347292
2012    8.537710  6.762489   6.570645
2013    8.566552  6.671812   6.315173
2014    8.951599  7.150001   6.764161
2015    9.168075  7.177207   6.975299
2016    9.145846  7.130726   7.038917
2017    8.649151  7.283064   7.602618
2018    8.921118  7.534107   7.840439
2019    8.980090  7.116679   7.614644
2020    9.163093  7.059414   7.048534
2021    8.155342  7.008156   7.139775
2022    8.943207  7.383220   7.048989
2023    9.110580  7.753725   7.954900
2024    9.511053  8.095272   8.603785
2025    9.494030  8.336463   8.760605
2026    9.247308  8.535956   9.100765

*** Phase Summary by Era:
  powerplay: 6.64 → 7.80 (+17.4%)
  middle: 6.84 → 7.57 (+10.6%)
  death: 8.61 → 9.03 (+4.8%)


In [15]:
# Visualize phase-wise scoring
fig = create_phase_scoring_chart(phase_data, output_path='figures/phase_scoring.png')
print("Created: figures/phase_scoring.png")

Created: figures/phase_scoring.png


In [16]:
fig

---
## Step 5: Validate Bowling Metrics (Claim: Economy +7.37%)

**Expected**: Economy rates increased from 7.96 to 8.55 (7.37% rise)

In [17]:
# Calculate bowling metrics
print("Calculating bowling metrics...")

bowling_data = clean_data.groupby('season').agg({
    'total_runs': 'sum',
    'is_legal': 'sum',
    'is_dot': 'sum'
}).reset_index()

bowling_data['economy_rate'] = bowling_data['total_runs'] / (bowling_data['is_legal'] / 6 + 0.001)
bowling_data['dot_ball_ratio'] = bowling_data['is_dot'] / (bowling_data['is_legal'] + 0.001)

print("\nBowling Metrics by Season:")
bowling_display = bowling_data[['season', 'economy_rate', 'dot_ball_ratio']].set_index('season')
print(bowling_display.to_string())

# Era comparison
early_econ = bowling_data[(bowling_data['season'] >= 2008) & (bowling_data['season'] <= 2015)]['economy_rate'].mean()
late_econ = bowling_data[(bowling_data['season'] >= 2016) & (bowling_data['season'] <= 2024)]['economy_rate'].mean()
econ_change = ((late_econ - early_econ) / early_econ) * 100

early_dot = bowling_data[(bowling_data['season'] >= 2008) & (bowling_data['season'] <= 2015)]['dot_ball_ratio'].mean()
late_dot = bowling_data[(bowling_data['season'] >= 2016) & (bowling_data['season'] <= 2024)]['dot_ball_ratio'].mean()
dot_change = ((late_dot - early_dot) / early_dot) * 100

print(f"\n*** Economy Rate: {early_econ:.2f} → {late_econ:.2f} (+{econ_change:.1f}%) (Claim: +7.37%)")
print(f"*** Dot Ball Ratio: {early_dot*100:.2f}% → {late_dot*100:.2f}% ({dot_change*100:.1f}%) (Claim: -2.59%)")

Calculating bowling metrics...

Bowling Metrics by Season:
        economy_rate  dot_ball_ratio
season                              
2008        7.978497        0.425161
2009        7.211374        0.430619
2010        7.814730        0.401986
2011        7.460410        0.421854
2012        7.582482        0.399786
2013        7.460635        0.418001
2014        7.943074        0.402937
2015        8.066067        0.402432
2016        8.028657        0.378618
2017        8.131291        0.379094
2018        8.358249        0.379252
2019        8.147286        0.388835
2020        8.001645        0.381414
2021        7.751955        0.397019
2022        8.171614        0.411288
2023        8.628335        0.378828
2024        9.111030        0.373911
2025        9.208096        0.365056
2026        9.350751        0.370484

*** Economy Rate: 7.69 → 8.26 (+7.4%) (Claim: +7.37%)
*** Dot Ball Ratio: 41.28% → 38.54% (-665.7%) (Claim: -2.59%)


In [18]:
# Visualize bowling metrics
fig_econ, fig_dot = create_bowling_metrics_chart(
    bowling_data, output_path='figures/bowling_metrics.png'
)
print("Created: figures/bowling_metrics_economy.png")
print("Created: figures/bowling_metrics_dotball.png")


Created: figures/bowling_metrics_economy.png
Created: figures/bowling_metrics_dotball.png


In [19]:
fig_econ

In [20]:
fig_dot

---
## Step 6: Validate Impact Player Rule (Claim: +1.4 sixes/match post-2023)

**Expected**: Impact Player rule added 1.4 sixes per match directly

In [21]:
# Compare pre/Impact Player rule era
print("Analyzing Impact Player Rule effect...")

pre_impact = season_sixes[(season_sixes['season'] >= 2021) & (season_sixes['season'] <= 2022)]
post_impact = season_sixes[(season_sixes['season'] >= 2023) & (season_sixes['season'] <= 2025)]

pre_sixes = pre_impact['sixes_per_match'].mean()
post_sixes = post_impact['sixes_per_match'].mean()
diff = post_sixes - pre_sixes

print("\n*** Pre-Impact Player (2021-2022):")
print(f"  Sixes/match: {pre_sixes:.2f}")
print(f"  Total sixes: {pre_impact['is_six'].sum()}")
print("\n*** Post-Impact Player (2023-2025):")
print(f"  Sixes/match: {post_sixes:.2f}")
print(f"  Total sixes: {post_impact['is_six'].sum()}")
print(f"\n*** Change: +{diff:.2f} sixes/match (Claim: +1.4)")

Analyzing Impact Player Rule effect...

*** Pre-Impact Player (2021-2022):
  Sixes/match: 12.90
  Total sixes: 1749

*** Post-Impact Player (2023-2025):
  Sixes/match: 16.85
  Total sixes: 3687

*** Change: +3.95 sixes/match (Claim: +1.4)


---
## Step 7: Analyze Venue Impact

**Expected**: Venues with <66m boundaries generate 50% more sixes

In [22]:
# Analyze venue impact
print("Analyzing venue impact on six-hitting...")

venue_data = clean_data.groupby('venue').agg({
    'is_six': 'sum',
    'match_id': 'first'
}).reset_index()

venue_matches = clean_data.groupby('venue').nunique()['match_id'].reset_index()
venue_matches.rename(columns={'match_id': 'matches'}, inplace=True)

venue_data = venue_data.merge(venue_matches, on='venue')
venue_data['sixes_per_match'] = venue_data['is_six'] / (venue_data['matches'] + 0.001)

venue_data = venue_data.sort_values('sixes_per_match', ascending=False)

print("\nTop 10 Venues by Sixes Per Match:")
print(venue_data[['venue', 'matches', 'is_six', 'sixes_per_match']].head(10).to_string(index=False))

# Identify short boundary venues
short_boundary_venues = ['M. Chinnaswamy Stadium', 'Bharat Ratna Shri Atal Bihari Vajpayee Ekana Cricket Stadium']
long_boundary_venues = ['Eden Gardens', 'M. A. Chidambaram Stadium']

if 'M. Chinnaswamy Stadium' in venue_data['venue'].values:
    chinnaswamy = venue_data[venue_data['venue'] == 'M. Chinnaswamy Stadium']['sixes_per_match'].values[0]
    print(f"\n*** M. Chinnaswamy (59.4m boundary): {chinnaswamy:.2f} sixes/match")

if 'M. A. Chidambaram Stadium' in venue_data['venue'].values:
    chepauk = venue_data[venue_data['venue'] == 'M. A. Chidambaram Stadium']['sixes_per_match'].values[0]
    print(f"*** M. A. Chidambaram (49.2% boundary %): {chepauk:.2f} sixes/match")

Analyzing venue impact on six-hitting...

Top 10 Venues by Sixes Per Match:
                                                                  venue  matches  is_six  sixes_per_match
Maharaja Yadavindra Singh International Cricket Stadium, New Chandigarh        5     129        25.794841
     Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium, Visakhapatnam        4      90        22.494376
                                       M Chinnaswamy Stadium, Bengaluru       24     478        19.915837
                                            Arun Jaitley Stadium, Delhi       28     527        18.820756
               Himachal Pradesh Cricket Association Stadium, Dharamsala        6     110        18.330278
                                                  Eden Gardens, Kolkata       27     490        18.147476
                   Rajiv Gandhi International Stadium, Uppal, Hyderabad       25     435        17.399304
                                                 Holkar Cricket Stadium     

In [23]:
# Visualize venue impact
fig = create_venue_impact_chart(venue_data, output_path='figures/venue_impact.png')
print("Created: figures/venue_impact.png")

Created: figures/venue_impact.png


In [24]:
fig

---
## Step 8: Perform Statistical Hypothesis Testing

**Expected**: 7 of 8 tests show significant results after FDR correction

In [25]:
# Perform statistical tests
print("Running statistical hypothesis tests...")

test_results = perform_statistical_tests(clean_data)

print("\nStatistical Test Results:")
for metric, results in test_results.items():
    print(f"\n  {metric}:")
    print(f"    Early mean: {results['early_mean']:.2f}")
    print(f"    Late mean: {results['late_mean']:.2f}")
    print(f"    p-value: {results['p_value']:.6f}")
    print(f"    Significant (p<0.05): {results['significant']}")

Running statistical hypothesis tests...

Statistical Test Results:

  strike_rate:
    Early mean: 120.80
    Late mean: 131.18
    p-value: 0.000000
    Significant (p<0.05): True

  sixes:


KeyError: 'early_mean'

In [ ]:
# Visualize statistical tests
fig = create_statistical_tests_chart(test_results, output_path='figures/statistical_tests.png')
print("Created: figures/statistical_tests.png")

In [ ]:
# Visualize q-values
fig = create_q_value_chart(test_results, corrected=True, output_path='figures/q_values.png')
print("Created: figures/q_values.png")

---
## Step 9: Generate Projections to 2028

**Expected**: Conservative (16-17), Moderate (19-20), Acceleration (22+) sixes/match by 2028

In [ ]:
# Calculate projections
print("Generating six-hitting projections to 2028...")

current_sixes = season_sixes[season_sixes['season'] == 2025]['sixes_per_match'].values[0]
print(f"Current sixes/match (2025): {current_sixes:.1f}")

projections = calculate_projected_sixes(
    current_sixes_per_match=current_sixes,
    annual_growth_rates=[0.03, 0.06, 0.09],  # 3%, 6%, 9% growth
    years_to_project=3  # 2025 to 2028
)

print("\nProjection Scenarios:")
for scenario, data in projections.items():
    name = 'Conservative' if '3' in scenario else ('Moderate' if '6' in scenario else 'Acceleration')
    print(f"  {name}: {data['final_sixes_per_match']:.1f} sixes/match")
    print(f"    Growth rate: {data['annual_growth_rate']*100:.0f}% annually")

In [ ]:
# Visualize projections
fig = create_projection_chart(projections, output_path='figures/projections.png')
print("Created: figures/projections.png")

---
## Summary and Validation Results

In [ ]:
# Final summary
print("=" * 80)
print("IPL ANALYSIS VALIDATION SUMMARY")
print("=" * 80)

results = {
    'Batting Strike Rate': {
        'found': round(early_sr, 2),
        'claim': '102.15',
        'change': round(change, 2)
    },
    'Six-Hitting': {
        'found': round(late_sixes, 1),
        'claim': '14.3',
        'growth': round(change, 1)
    },
    'Economy Rate': {
        'found': round(econ_change, 1),
        'claim': '+7.37%'
    },
    'Dot Ball Ratio': {
        'found': round(dot_change, 1),
        'claim': '-2.59%'
    },
    'Impact Player Rule': {
        'found': round(diff, 2),
        'claim': '+1.4 sixes/match'
    }
}

for metric, data in results.items():
    print(f"\n{metric}:")
    print(f"  Found: {data['found']}")
    print(f"  Claim: {data['claim']}")
    print(f"  {'✓ Validated' if abs(data['found'] - data['claim']) < 5 else '✗ Partially validated'}")

print("\n" + "=" * 80)
print("All figures saved to figures/ directory:")
print("  - strike_rate_trend.png")
print("  - sixes_growth.png")
print("  - phase_scoring.png")
print("  - bowling_metrics.png")
print("  - venue_impact.png")
print("  - statistical_tests.png")
print("  - q_values.png")
print("  - projections.png")
print("=" * 80)

---
## Save Results

Save summary statistics to JSON for programmatic access.

In [ ]:
import json
import datetime

# Create summary JSON
summary = {
    'analysis_date': datetime.datetime.now().isoformat(),
    'data_summary': {
        'total_records': len(clean_data),
        'total_matches': clean_data['match_id'].nunique(),
        'total_sixes': clean_data['is_six'].sum(),
        'seasons_covered': list(range(2008, 2026))
    },
    'era_comparison': {
        'early_sr': round(early_sr, 2),
        'late_sr': round(late_sr, 2),
        'sr_change': round(change, 2),
        'early_sixes': round(early_sixes, 2),
        'late_sixes': round(late_sixes, 2),
        'sixes_change': round(change, 1),
        'early_economy': round(early_econ, 2),
        'late_economy': round(late_econ, 2),
        'economy_change': round(econ_change, 2),
        'early_dot': round(early_dot * 100, 2),
        'late_dot': round(late_dot * 100, 2),
        'dot_change': round(dot_change, 1)
    },
    'impact_player_effect': {
        'pre_impact_sixes': round(pre_sixes, 2),
        'post_impact_sixes': round(post_sixes, 2),
        'change': round(diff, 2)
    },
    'statistical_tests': {
        'tests_passed': sum(1 for t in test_results.values() if t['significant']),
        'tests_total': len(test_results)
    },
    'projections': projections
}

# Save
output_path = Path('results/summary.json')
output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, 'w') as f:
    json.dump(summary, f, indent=2, default=str)

print(f"Summary saved to: {output_path}")
print(json.dumps(summary, indent=2))